# Retail Lakehouse - Exploration

Quick EDA over the gold layer produced by the pipeline. Run `python src/run_pipeline.py` first.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
from config import GOLD_DIR, p
from spark_session import get_spark
spark = get_spark('exploration')
for v in ['fact_sales','kpi_cltv','kpi_country','kpi_top_products','kpi_overview']:
    spark.read.format('delta').load(p(GOLD_DIR / v)).createOrReplaceTempView(v)
print('views ready')

In [ ]:
spark.sql('SELECT * FROM kpi_overview').toPandas()

In [ ]:
# Revenue by country
pdf = spark.sql('SELECT country, revenue FROM kpi_country ORDER BY revenue DESC').toPandas()
pdf.plot.bar(x='country', y='revenue', legend=False, title='Revenue by country');

In [ ]:
# Monthly revenue trend
spark.sql('''
  SELECT date_format(date,'yyyy-MM') AS month, round(sum(line_amount),2) AS revenue
  FROM fact_sales WHERE is_cancellation = false
  GROUP BY 1 ORDER BY 1
''').toPandas()